# NumCompute quickstart

An end-to-end tour of the NumCompute toolkit (plain Python + NumPy) on real weather data: daily observations for **Adelaide in 2015** from the Australian Bureau of Meteorology

The file has missing cells, two categorical columns, many tied values and a yes/no label (`RainTomorrow`), so every module gets tested.

Modules that are being tested:
1. Load the CSV - 'io'
2. Preprocessing and pipelines - 'preprocessing', 'pipeline'
3. Sorting, top-k and searching - 'sort_search'
4. Ranking and percentiles - 'rank'
5. Descriptive and streaming statistics - 'stats'
6. Evaluation metrics - 'metrics'
7. Finite-difference gradients - 'optim'
8. Loops vs vectorised benchmarks - 'benchmarking'

In [1]:
from pathlib import Path

import numpy as np

from numcompute.io import load_csv, load_csv_chunks, read_header
from numcompute.pipeline import FeatureUnion, Pipeline, ThresholdClassifier
from numcompute.preprocessing import MinMaxScaler, OneHotEncoder, SimpleImputer, StandardScaler
from numcompute.rank import percentile, rank
from numcompute.sort_search import binary_search, multi_key_sort, quickselect, stable_sort, topk
from numcompute import stats

np.set_printoptions(precision=3, suppress=True)

# Works whether the notebook is started from demo/ or from the repo root.
DATA = Path("../data/adelaide_weather_2015.csv")
if not DATA.exists():
    DATA = Path("data/adelaide_weather_2015.csv")
print("Reading", DATA)

Reading ../data/adelaide_weather_2015.csv


### 1. Load the CSV - 'io'

In [2]:
header = read_header(DATA)
print(header)

NUM_COLS = ["MinTemp", "MaxTemp", "Rainfall", "WindGustSpeed", "Humidity3pm",
            "Pressure3pm", "Temp3pm"]
CAT_COLS = ["WindGustDir", "RainToday"]
num_idx = [header.index(c) for c in NUM_COLS]

X_num = load_csv(DATA, skip_header=1, usecols=num_idx)
X_cat = load_csv(DATA, skip_header=1, usecols=[header.index(c) for c in CAT_COLS], dtype=str)
dates = load_csv(DATA, skip_header=1, usecols=0, dtype=str).ravel()
y = (load_csv(DATA, skip_header=1, usecols=header.index("RainTomorrow"), dtype=str).ravel()
     == "Yes").astype(int)

print("numeric:", X_num.shape, "categorical:", X_cat.shape, "labels:", y.shape)
print("rainy tomorrow on", y.sum(), "of", y.size, "days")
print("first 3 numeric rows:\n", X_num[:3])

['Date', 'MinTemp', 'MaxTemp', 'Rainfall', 'WindGustDir', 'WindGustSpeed', 'Humidity3pm', 'Pressure3pm', 'Temp3pm', 'RainToday', 'RainTomorrow']
numeric: (360, 7) categorical: (360, 2) labels: (360,)
rainy tomorrow on 74 of 360 days
first 3 numeric rows:
 [[  15.1   37.     0.    31.    20.  1009.3   34.2]
 [  23.2   44.1    0.    41.    10.  1006.7   42.8]
 [  24.4   38.2    0.    56.    17.  1007.3   35.4]]


In [3]:
print("missing numeric cells:", dict(zip(NUM_COLS, np.isnan(X_num).sum(axis=0).tolist())))
print("missing text cells:   ", dict(zip(CAT_COLS, (X_cat == "").sum(axis=0).tolist())))

gap_rows = np.flatnonzero(np.isnan(X_num).any(axis=1))
print("days with a missing number:", dates[gap_rows])

missing numeric cells: {'MinTemp': 1, 'MaxTemp': 1, 'Rainfall': 3, 'WindGustSpeed': 4, 'Humidity3pm': 2, 'Pressure3pm': 4, 'Temp3pm': 2}
missing text cells:    {'WindGustDir': 4, 'RainToday': 3}
days with a missing number: ['2015-03-11' '2015-03-12' '2015-04-15' '2015-05-18' '2015-05-21'
 '2015-05-25' '2015-11-21' '2015-12-12']


In [4]:
# Missing values can also be filled with a placeholder, or the incomplete rows can be dropped:

filled = load_csv(DATA, skip_header=1, usecols=num_idx, fill_value=-1)
skipped = load_csv(DATA, skip_header=1, usecols=num_idx, missing="skip")
first_gap = gap_rows[0]
print(f"fill_value=-1 -> {filled.shape} | {dates[first_gap]}:", filled[first_gap])
print("missing='skip' ->", skipped.shape, "(incomplete days dropped)")

fill_value=-1 -> (360, 7) | 2015-03-11: [12.4 25.   0.  37.  44.  -1.  23.9]
missing='skip' -> (352, 7) (incomplete days dropped)


In [5]:
# For files too big for memory, `load_csv_chunks` streams the rows in blocks:

sizes = [chunk.shape[0] for chunk in
         load_csv_chunks(DATA, chunk_size=100, skip_header=1, usecols=num_idx)]
print("rows per chunk:", sizes, "| total:", sum(sizes))

rows per chunk: [100, 100, 100, 60] | total: 360


### 2. Preprocessing and pipelines ('preprocessing', 'pipeline')

In [6]:
imputer = SimpleImputer(strategy="median")
X_imp = imputer.fit_transform(X_num)
print("median used per column:", dict(zip(NUM_COLS, imputer.statistics_.round(1).tolist())))
print("NaNs left after imputing:", np.isnan(X_imp).sum())

median used per column: {'MinTemp': 12.1, 'MaxTemp': 21.7, 'Rainfall': 0.0, 'WindGustSpeed': 33.0, 'Humidity3pm': 44.0, 'Pressure3pm': 1018.3, 'Temp3pm': 20.4}
NaNs left after imputing: 0


In [7]:
scaler = StandardScaler().fit(X_imp)
X_std = scaler.transform(X_imp)
print("column means before:", scaler.mean_)
print("means after :", X_std.mean(axis=0))
print("stds after  :", X_std.std(axis=0))

X_mm = MinMaxScaler(feature_range=(0, 1)).fit_transform(X_imp)
print("min-max range:", X_mm.min(axis=0), "to", X_mm.max(axis=0))

column means before: [  12.484   23.209    1.098   33.953   44.189 1018.658   21.87 ]
means after : [-0.  0. -0. -0.  0.  0. -0.]
stds after  : [1. 1. 1. 1. 1. 1. 1.]
min-max range: [0. 0. 0. 0. 0. 0. 0.] to [1. 1. 1. 1. 1. 1. 1.]


In [8]:
# Text columns: fill the empty cells with the most common value, then one-hot encode.

cat_imputer = SimpleImputer(strategy="most_frequent", missing_values="")
X_cat_filled = cat_imputer.fit_transform(X_cat)
print("filled with:", dict(zip(CAT_COLS, cat_imputer.statistics_.tolist())))

encoder = OneHotEncoder()
X_onehot = encoder.fit_transform(X_cat_filled)
names = encoder.get_feature_names_out(CAT_COLS)
print(len(names), "output columns:", names.tolist())
print("first day is 1 in:", names[X_onehot[0] == 1].tolist())

filled with: {'WindGustDir': 'WSW', 'RainToday': 'No'}
18 output columns: ['WindGustDir_E', 'WindGustDir_ENE', 'WindGustDir_ESE', 'WindGustDir_N', 'WindGustDir_NE', 'WindGustDir_NNE', 'WindGustDir_NNW', 'WindGustDir_NW', 'WindGustDir_S', 'WindGustDir_SE', 'WindGustDir_SSE', 'WindGustDir_SSW', 'WindGustDir_SW', 'WindGustDir_W', 'WindGustDir_WNW', 'WindGustDir_WSW', 'RainToday_No', 'RainToday_Yes']
first day is 1 in: ['WindGustDir_WNW', 'RainToday_No']


In [9]:
# A `Pipeline` chains the steps. This is the example from the assignment spec:

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("encode", OneHotEncoder()),
])
rain_today = (X_cat_filled[:, [1]] == "Yes").astype(float)
X_tr = pipe.fit_transform(rain_today)
print(pipe)
print("output shape:", X_tr.shape, "| first rows:\n", X_tr[:3])

Pipeline(steps=[('scale', StandardScaler(with_mean=True, with_std=True)), ('encode', OneHotEncoder(handle_unknown='error'))])
output shape: (360, 2) | first rows:
 [[1. 0.]
 [1. 0.]
 [1. 0.]]


In [10]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("features", FeatureUnion([("std", StandardScaler()), ("minmax", MinMaxScaler())])),
])
X_both = numeric_pipe.fit_transform(X_num)
print("FeatureUnion output:", X_both.shape, "(7 standardised + 7 min-max columns)")
print("step names:", list(numeric_pipe.named_steps))

FeatureUnion output: (360, 14) (7 standardised + 7 min-max columns)
step names: ['impute', 'features']


In [11]:
#A pipeline can also end in an estimator. `ThresholdClassifier` is a tiny example model: it predicts rain tomorrow from afternoon humidity alone.

model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("clf", ThresholdClassifier(feature=NUM_COLS.index("Humidity3pm"))),
])
y_pred = model.fit(X_num, y).predict(X_num)
print(f"rule learned: rain tomorrow if 3pm humidity > {model['clf'].threshold_:.1f}%")
print("first 15 days  true:", y[:15])
print("               pred:", y_pred[:15])

rule learned: rain tomorrow if 3pm humidity > 50.4%
first 15 days  true: [0 0 0 0 0 0 1 1 1 1 0 1 1 0 0]
               pred: [0 0 0 0 0 0 0 1 1 1 1 1 1 1 0]


### 3. Sorting, top-k and searching ('sort_search')

In [12]:
max_temp = X_imp[:, NUM_COLS.index("MaxTemp")]
values, idx = topk(max_temp, k=3, largest=True)
print("3 hottest days:", list(zip(dates[idx].tolist(), values.tolist())))

rain = X_num[:, NUM_COLS.index("Rainfall")]
values, idx = topk(rain, k=3, largest=True)  # NaNs are never picked before real values
print("3 wettest days:", list(zip(dates[idx].tolist(), values.tolist())))

3 hottest days: [('2015-01-02', 44.1), ('2015-12-19', 43.2), ('2015-12-17', 42.9)]
3 wettest days: [('2015-05-19', 31.6), ('2015-09-17', 22.4), ('2015-07-11', 18.6)]


In [13]:
humidity = X_num[:, NUM_COLS.index("Humidity3pm")]
known = humidity[~np.isnan(humidity)]
k = known.size // 2
print(f"quickselect (k={k}):", quickselect(known, k))
print("same from a full sort:", np.sort(known)[k])

quickselect (k=179): 44.0
same from a full sort: 44.0


In [14]:
sorted_hum = stable_sort(known)
for value in [50, 71]:  # 50% occurs in 2015, 71% never does
    pos, found = binary_search(sorted_hum, value)
    print(f"humidity {value}% -> insertion index {pos}, present: {found}")

humidity 50% -> insertion index 228, present: True
humidity 71% -> insertion index 325, present: False


In [15]:
# Wettest days first, ties broken by the hotter day. Missing rainfall sorts last.
day_no = np.arange(len(dates))
table = np.column_stack([day_no, np.nan_to_num(rain, nan=-1), max_temp])
ordered = multi_key_sort(table, keys=[1, 2], descending=[True, True])
for d, r, t in ordered[:5]:
    print(dates[int(d)], f"rain {r:5.1f} mm, max {t:4.1f} C")

2015-05-19 rain  31.6 mm, max 19.7 C
2015-09-17 rain  22.4 mm, max 17.4 C
2015-07-11 rain  18.6 mm, max 11.8 C
2015-01-13 rain  16.8 mm, max 20.5 C
2015-04-25 rain  13.0 mm, max 16.4 C


### 4. Ranking and percentiles ('rank')

Humidity is a whole number, so ties are common; this is where the methods differ.

In [16]:
sample = humidity[47:57]  # 10 days in Feb with repeated humidity values
print("dates   :", dates[47], "to", dates[56])
print("humidity:", sample)
for method in ["average", "dense", "ordinal"]:
    print(f"{method:8s}:", rank(sample, method=method))

dates   : 2015-02-17 to 2015-02-26
humidity: [43. 31. 17. 21. 20. 20. 43. 31. 28. 20.]
average : [9.5 7.5 1.  5.  3.  3.  9.5 7.5 6.  3. ]
dense   : [6. 5. 1. 3. 2. 2. 6. 5. 4. 2.]
ordinal : [ 9.  7.  1.  5.  2.  3. 10.  8.  6.  4.]


In [17]:
# Most days have zero rain, which makes the interpolation choice visible:

for q in [50, 75, 90, 95]:
    row = {m: round(float(percentile(rain, q, interpolation=m)), 2)
           for m in ["linear", "lower", "higher", "midpoint"]}
    print(f"rainfall p{q}:", row)

rainfall p50: {'linear': 0.0, 'lower': 0.0, 'higher': 0.0, 'midpoint': 0.0}
rainfall p75: {'linear': 0.6, 'lower': 0.6, 'higher': 0.6, 'midpoint': 0.6}
rainfall p90: {'linear': 3.28, 'lower': 3.2, 'higher': 3.4, 'midpoint': 3.3}
rainfall p95: {'linear': 5.8, 'lower': 5.8, 'higher': 5.8, 'midpoint': 5.8}


### 5. Descriptive and streaming statistics ('stats')

Axis-aware statistics ignore NaNs by default: 'axis=0' gives one value per column.

In [18]:
for name, fn in [("mean", stats.mean), ("median", stats.median), ("std", stats.std),
                 ("min", stats.minimum), ("max", stats.maximum)]:
    print(f"{name:6s}", dict(zip(NUM_COLS, np.round(fn(X_num, axis=0), 1).tolist())))

mean   {'MinTemp': 12.5, 'MaxTemp': 23.2, 'Rainfall': 1.1, 'WindGustSpeed': 34.0, 'Humidity3pm': 44.2, 'Pressure3pm': 1018.7, 'Temp3pm': 21.9}
median {'MinTemp': 12.1, 'MaxTemp': 21.7, 'Rainfall': 0.0, 'WindGustSpeed': 33.0, 'Humidity3pm': 44.0, 'Pressure3pm': 1018.3, 'Temp3pm': 20.4}
std    {'MinTemp': 5.2, 'MaxTemp': 7.7, 'Rainfall': 3.2, 'WindGustSpeed': 10.1, 'Humidity3pm': 18.8, 'Pressure3pm': 7.2, 'Temp3pm': 7.5}
min    {'MinTemp': 1.8, 'MaxTemp': 11.4, 'Rainfall': 0.0, 'WindGustSpeed': 13.0, 'Humidity3pm': 4.0, 'Pressure3pm': 998.8, 'Temp3pm': 9.1}
max    {'MinTemp': 30.7, 'MaxTemp': 44.1, 'Rainfall': 31.6, 'WindGustSpeed': 67.0, 'Humidity3pm': 95.0, 'Pressure3pm': 1036.8, 'Temp3pm': 42.8}


In [19]:
counts, edges = stats.histogram(X_num[:, NUM_COLS.index("MaxTemp")], bins=8)
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"{lo:5.1f} - {hi:5.1f} C | {'#' * int(c // 3)} {c}")

print("max-temp quartiles:", stats.quantile(X_num[:, NUM_COLS.index("MaxTemp")], [0.25, 0.5, 0.75]))

 11.4 -  15.5 C | ################## 54
 15.5 -  19.6 C | ############################ 86
 19.6 -  23.7 C | ###################### 67
 23.7 -  27.8 C | ################## 56
 27.8 -  31.8 C | ############# 41
 31.8 -  35.9 C | ######## 24
 35.9 -  40.0 C | ###### 20
 40.0 -  44.1 C | ### 11
max-temp quartiles: [16.65 21.7  28.2 ]


In [20]:
# 'RunningStats' (Welford) gets the same answer chunk by chunk, without holding all the data:
col = header.index("MaxTemp")
running = stats.RunningStats()
for chunk in load_csv_chunks(DATA, chunk_size=50, skip_header=1, usecols=col):
    running.update(chunk.ravel())

full = X_num[:, NUM_COLS.index("MaxTemp")]
print("streamed :", running.count, "days, mean", round(running.mean, 4),
      "variance", round(running.variance, 4))
print("all at once:", int((~np.isnan(full)).sum()), "days, mean", round(stats.mean(full), 4),
      "variance", round(stats.var(full), 4))

streamed : 359 days, mean 23.2128 variance 59.3446
all at once: 359 days, mean 23.2128 variance 59.3446


### 6. Evaluation metrics (`metrics`)


In [21]:
from numcompute.metrics import (accuracy, auc, confusion_matrix, f1, mae, mse, precision,
                                r2, recall, rmse, roc_curve)

print("accuracy :", round(accuracy(y, y_pred), 3))
print("precision:", round(precision(y, y_pred), 3))
print("recall   :", round(recall(y, y_pred), 3))
print("f1       :", round(f1(y, y_pred), 3))
print("macro f1 :", round(f1(y, y_pred, average="macro"), 3))
print("confusion matrix:\n", confusion_matrix(y, y_pred))

accuracy : 0.767
precision: 0.458
recall   : 0.743
f1       : 0.567
macro f1 : 0.704
confusion matrix:
 [[221  65]
 [ 19  55]]


In [22]:
# For ROC/AUC we need a score instead of a hard 0/1 prediction. Afternoon humidity works as a score on its own: higher humidity should mean rain is more likely.

score = X_imp[:, NUM_COLS.index("Humidity3pm")]
fpr, tpr, thresholds = roc_curve(y, score)
print("ROC points:", len(fpr), "| AUC using humidity:", round(auc(fpr, tpr), 3))

# Compare with a weaker signal
score_temp = -X_imp[:, NUM_COLS.index("MaxTemp")]  # cooler days -> rain more likely
fpr_t, tpr_t, _ = roc_curve(y, score_temp)
print("AUC using (negative) max temperature:", round(auc(fpr_t, tpr_t), 3))

ROC points: 79 | AUC using humidity: 0.809
AUC using (negative) max temperature: 0.71


In [23]:
#Regression metrics, using 'MeanRegressor' as a baseline that always predicts the average 3pm temperature. A baseline like this should score R² = 0.

from numcompute.pipeline import MeanRegressor

temp3pm = X_imp[:, NUM_COLS.index("Temp3pm")]
baseline = MeanRegressor().fit(X_imp, temp3pm).predict(X_imp)
print(f"MSE {mse(temp3pm, baseline):.2f} | MAE {mae(temp3pm, baseline):.2f} | "
      f"RMSE {rmse(temp3pm, baseline):.2f} | R2 {r2(temp3pm, baseline):.2f}")

MSE 56.49 | MAE 6.15 | RMSE 7.52 | R2 0.00


### 7. Finite-difference gradients ('optim')



In [24]:
from numcompute.optim import grad, jacobian

f = lambda v: v[0] ** 2 * v[1] + np.sin(v[1])
x0 = np.array([1.5, 0.7])
exact = np.array([2 * x0[0] * x0[1], x0[0] ** 2 + np.cos(x0[1])])
central = grad(f, x0)
forward = grad(f, x0, method="forward")
print("exact  :", exact)
print("central:", central, "| max error", f"{np.abs(central - exact).max():.1e}")
print("forward:", forward, "| max error", f"{np.abs(forward - exact).max():.1e}")

exact  : [2.1   3.015]
central: [2.1   3.015] | max error 3.0e-11
forward: [2.1   3.015] | max error 7.0e-06


In [25]:
A = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
J = jacobian(lambda v: A @ v, [0.5, -1.0])
print(J)
print("matches A:", np.allclose(J, A))

[[1. 2.]
 [3. 4.]
 [5. 6.]]
matches A: True


In [26]:
x_feat = X_imp[:, NUM_COLS.index("MaxTemp")]
loss = lambda w: np.mean((w[0] * x_feat + w[1] - temp3pm) ** 2)

a, b = np.polyfit(x_feat, temp3pm, 1)
print("gradient at (0, 0)          :", grad(loss, [0.0, 0.0]))
print(f"gradient at best fit ({a:.2f}, {b:.2f}):", grad(loss, [a, b]))

gradient at (0, 0)          : [-1129.278   -43.741]
gradient at best fit (0.96, -0.50): [0. 0.]


### 8. Loops vs vectorised benchmarks ('benchmarking')

In [27]:
from IPython.display import Markdown

BENCH = Path("../benchmark/results.md")
if not BENCH.exists():
    BENCH = Path("benchmark/results.md")
Markdown(BENCH.read_text())

# Benchmark results

Best of repeated runs (seconds per call), seed 42.

| Operation | n | Loop (s) | Vectorised (s) | Speed-up |
| --- | ---: | ---: | ---: | ---: |
| StandardScaler | 1,000 | 0.00173 | 0.00022 | 7.9x |
| topk (k=10) | 1,000 | 0.00006 | 0.00004 | 1.4x |
| rank (average) | 1,000 | 0.00029 | 0.00019 | 1.5x |
| Welford mean/var | 1,000 | 0.00020 | 0.00005 | 4.4x |
| euclidean_distances (n x 10) | 1,000 | 0.00736 | 0.00010 | 70.7x |
| StandardScaler | 10,000 | 0.01933 | 0.00436 | 4.4x |
| topk (k=10) | 10,000 | 0.00087 | 0.00009 | 9.6x |
| rank (average) | 10,000 | 0.00429 | 0.00140 | 3.1x |
| Welford mean/var | 10,000 | 0.00216 | 0.00008 | 27.7x |
| euclidean_distances (n x 10) | 10,000 | 0.10822 | 0.00467 | 23.2x |
| StandardScaler | 100,000 | 0.22679 | 0.03964 | 5.7x |
| topk (k=10) | 100,000 | 0.00518 | 0.00155 | 3.3x |
| rank (average) | 100,000 | 0.07662 | 0.01972 | 3.9x |
| Welford mean/var | 100,000 | 0.01053 | 0.00189 | 5.6x |
| euclidean_distances (n x 10) | 100,000 | 0.82591 | 0.03183 | 25.9x |
| grad (central) | 100 | 0.00170 | 0.00023 | 7.4x |
| grad (central) | 500 | 0.01535 | 0.01424 | 1.1x |
| grad (central) | 2,000 | 0.21184 | 0.27045 | 0.8x |

Environment: python 3.14.5, numpy 2.5.4, os Windows 11, machine AMD64, processor Intel64 Family 6 Model 170 Stepping 4, GenuineIntel, cpu_count 18


In [28]:
from numcompute.benchmarking import compare, environment_info

big = np.tile(X_imp, (300, 1))  # 108,000 rows x 7 columns

def loop_standardise(X):
    out = np.empty_like(X)
    for j in range(X.shape[1]):
        col = X[:, j]
        mean = sum(col) / len(col)
        std = (sum((v - mean) ** 2 for v in col) / len(col)) ** 0.5
        out[:, j] = [(v - mean) / std for v in col]
    return out

rows = compare({"python loops": lambda: loop_standardise(big),
                "StandardScaler": lambda: StandardScaler().fit_transform(big)},
               repeats=3, number=1)
for r in rows:
    print(f"{r['name']:15s} {r['mean'] * 1000:8.1f} ms  ({r['speedup']:.1f}x)")
print(environment_info())

StandardScaler      15.7 ms  (23.0x)
python loops       361.4 ms  (1.0x)
{'python': '3.11.7', 'numpy': '2.4.6', 'os': 'Darwin 27.0.0', 'machine': 'arm64', 'processor': 'arm', 'cpu_count': '8'}
